# Point-wise profile confidence bands

復元フラックス $x(v_{\min})$ の **点ごと信頼区間**(point-wise profile confidence band)を計算・保存・閲覧するノートブック。

## 方式(kyphys-creator/neutrinoAnalysis の移植)

走査する各 $v_{\min}$ 点 $j$ で:

1. フラックス段 $x_j$ を試行値 $v$ に**固定**して残りを再フィット → $\chi^2_{\rm fixed}$。自由フィットの $\chi^2_{\rm free}$ との差が観測 $\Delta\chi^2_{\rm obs}$。
2. **固定フィットモデル**から Poisson 疑似実験を生成し、各疑似データを自由/固定で再フィットして $\Delta\chi^2$ 分布を作る。その CL 分位点がカットオフ(単調・非負制約のため $\chi^2(1)$ は仮定しない)。
3. $\Delta\chi^2_{\rm obs} <$ カットオフ なら $v$ はバンド内。端は外向きブラケット+幾何二分法(共通乱数)で決める。

実装は [`quantum_sensor.statistics.find_confidence_band`](src/quantum_sensor/statistics.py)。ソルバーは CLARABEL(この QP では OSQP の ~40 倍速、$\chi^2$ 一致 ~3e-5)。

## 出力(各ランフォルダ `results/<DET>/bkg-<bkg>/<config>/` 内)

| ファイル | 内容 |
|---|---|
| `flux_profile_band.csv` | `vmin_mid, best_fit, lo68, hi68, lo95, hi95`(自然単位) |
| `flux_profile_band.json` | 点ごとの全記録(カットオフ・評価回数つき) |
| `flux_profile_band.pdf` | η + best-fit 階段 + 塗りバンドの図 |

**所要時間の目安**: 1点 30秒〜3分 × 約12点/設定。§3 のフル計算は重いので、まず §2 の1点クイック試行で感触を見ること。


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantum_sensor import DarkMatterQuantumAnalysis, RunConfig
from quantum_sensor.statistics import find_confidence_band, pointwise_band
from quantum_sensor.plotting import (plot_flux_with_pointwise_bands,
                                     run_dir, RESULTS_DIR, ETA_TO_CM_INV)

LEVELS = (0.68, 0.954)


def band_table(bands):
    """pointwise_band の結果を、相対誤差つきの読みやすい表にする。"""
    rows = []
    for b in bands:
        lo68, hi68 = b['band'][LEVELS[0]]
        lo95, hi95 = b['band'][LEVELS[1]]
        f = b['best_fit']
        rows.append({
            'index': b['index'], 'vmin [km/s]': round(b['vmin_mid'], 1),
            'best fit': f, 'lo68': lo68, 'hi68': hi68, 'lo95': lo95, 'hi95': hi95,
            '-68%': f'-{(1 - lo68 / f) * 100:.0f}%' if f > 0 else '-',
            '+68%': f'+{(hi68 / f - 1) * 100:.0f}%' if f > 0 and np.isfinite(hi68) else '-',
            'evals': b['n_evaluations'],
        })
    return pd.DataFrame(rows)


def save_band_products(a, bands):
    """CSV / JSON / PDF をランフォルダに保存(run_pointwise_bands.py と同形式)。"""
    out = run_dir(a)
    out.mkdir(parents=True, exist_ok=True)
    table = np.array([[b['vmin_mid'], b['best_fit'],
                       b['band'][LEVELS[0]][0], b['band'][LEVELS[0]][1],
                       b['band'][LEVELS[1]][0], b['band'][LEVELS[1]][1]]
                      for b in bands])
    np.savetxt(out / 'flux_profile_band.csv', table, delimiter=',', comments='',
               header='vmin_mid,best_fit,lo68,hi68,lo95,hi95')
    with open(out / 'flux_profile_band.json', 'w') as f:
        json.dump([{**b, 'levels': list(b['levels']),
                    'band': {str(k): list(v) for k, v in b['band'].items()}}
                   for b in bands], f, indent=1)
    plot_flux_with_pointwise_bands(a, bands)
    print(f'saved 3 files under {out}')


## 1. 設定を選んでフィット

`material`(`'Al'`=TES / `'TiN'`=MKID)、`q`(`'0'`=重い媒介子 / `'2'`=軽い)、
`mass`(`'1'`=10 MeV / `'2'`=100 MeV / `'3'`=1 GeV)、`nbins`(Al: 5/10, TiN: 5/9)。


In [ ]:
CONFIG = RunConfig(material='TiN', q='0', mass='2', nbins=5,
                   eta='Halo', background='none')

a = DarkMatterQuantumAnalysis(CONFIG)
a.optimize()

print(f'config        : {CONFIG}')
print(f'v_min columns : {a.n_vmin}  (window {a.vmin_mid[0]:.0f}-{a.vmin_mid[-1]:.0f} km/s)')
print(f'energy bins   : {a.n_ebins}')
print(f'counts per bin: {np.array2string(a.observed, precision=1)}')
print(f'best-fit chi2 : {a.result.fun:.2e}  (self-consistent -> ~0)')


## 2. まず1点だけ試す(クイック、〜1分)

感触を見る用の軽い設定(`num_pseudo=20, n_pseudo_edge=50, rel_tol=0.05`)。
**この設定は試行用で、論文の数字には §3 を使うこと。**


In [ ]:
idx = a.n_vmin // 3   # 窓の中ほどの点

b1 = find_confidence_band(a, index=idx, levels=LEVELS,
                          num_pseudo=20, n_pseudo_edge=50, rel_tol=0.05)

f, (lo, hi) = b1['best_fit'], b1['band'][LEVELS[0]]
print(f"v_min = {b1['vmin_mid']:.0f} km/s")
print(f'best fit = {f:.3g}   (input eta = {a.eta[idx]:.3g})')
for lv in LEVELS:
    lo, hi = b1['band'][lv]
    print(f'{lv*100:.0f}% : [{lo:.3g}, {hi:.3g}]'
          + (f'   (-{(1-lo/f)*100:.0f}% / +{(hi/f-1)*100:.0f}%)' if f > 0 else ''))


## 3. フルバンド(論文設定、〜30分/設定)

走査点はデフォルトで「フラックスが正の窓」をインデックス対数等間隔に 12 点
(低速側に密)。点を増やす/場所を指定するなら `n_indices=` / `indices=[...]`。

既に計算済みなら §4 で読み込むだけでよい(再計算不要)。


In [ ]:
bands = pointwise_band(a, n_indices=12, levels=LEVELS,
                       num_pseudo=30, n_pseudo_edge=200, rel_tol=0.02)

save_band_products(a, bands)
band_table(bands)


## 4. 保存済みバンドの読み込み・表示(再計算なし)

`flux_profile_band.csv` から表と図を再構成する。設定を変えたら §1 だけ実行し直せばよい。


In [ ]:
def load_band_csv(analysis):
    path = run_dir(analysis) / 'flux_profile_band.csv'
    df = pd.read_csv(path)
    f = df['best_fit']
    df['-68%'] = ((1 - df['lo68'] / f) * 100).round(0).astype('Int64').astype(str) + '%'
    df['+68%'] = ((df['hi68'] / f - 1) * 100).round(0).astype('Int64').astype(str) + '%'
    return df

df = load_band_csv(a)
display(df)

# 図も CSV から再構成(fill は 68/95 の2レベル)
fig, ax = plt.subplots(figsize=(8, 6))
for lv, (lo, hi), alpha in [('95.4%', ('lo95', 'hi95'), 0.18), ('68%', ('lo68', 'hi68'), 0.35)]:
    ax.fill_between(df['vmin_mid'], df[lo] * ETA_TO_CM_INV, df[hi] * ETA_TO_CM_INV,
                    alpha=alpha, color='C0', label=f'{lv} band')
ax.hlines(a.flux * ETA_TO_CM_INV, a.rm.vmin_low, a.rm.vmin_high, color='C0', lw=1.5, label='Best-Fit')
vg = np.logspace(0, np.log10(800), 400)
from quantum_sensor.eta_models import eta as eta_model
from quantum_sensor.constants import DM_MASS
ax.plot(vg, eta_model(CONFIG.eta, DM_MASS[CONFIG.mass], vg) * ETA_TO_CM_INV,
        'r-', lw=2, label=r'input $\eta$')
ax.set_xscale('log'); ax.set_xlim(1, 800)
ax.set_xlabel(r'$v_{min}$ [km/s]'); ax.set_ylabel(r'$\tilde{\eta}$ [cm$^{-1}$]')
ax.legend(); ax.grid(True, which='both', ls='--', alpha=0.4)
ax.set_title(f'{CONFIG.material} q{CONFIG.q} M{CONFIG.mass} R{CONFIG.nbins} (from saved CSV)');


## 5. 全設定のサマリー一覧

`results/` 以下にある全ての `flux_profile_band.csv` を集計。
「68% 半幅の中央値」はバンドのタイトさの目安(best fit に対する %)。


In [ ]:
rows = []
for path in sorted(RESULTS_DIR.glob('*/bkg-*/*/flux_profile_band.csv')):
    df = pd.read_csv(path)
    ok = df['best_fit'] > 0
    half = ((df.loc[ok, 'hi68'] - df.loc[ok, 'lo68']) / 2 / df.loc[ok, 'best_fit'] * 100)
    rows.append({
        'run': path.parent.name,
        'detector': path.parts[-4], 'bkg': path.parts[-3].replace('bkg-', ''),
        'points': len(df),
        'v range [km/s]': f"{df['vmin_mid'].min():.0f}-{df['vmin_mid'].max():.0f}",
        'median 68% half-width': f'{half.median():.0f}%',
        'lower edge hits 0': bool((df['lo68'] == 0).any()),
    })
summary = pd.DataFrame(rows)
display(summary)
print(f'{len(summary)} band runs found under {RESULTS_DIR}')
